In [2]:
# pip install jupyterlab pandas sqlalchemy "psycopg[binary]" python-dotenv

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

True

In [ ]:

database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise ValueError("DATABASE_URL missing .")

engine = create_engine(database_url, pool_pre_ping=True)

# Read-only session: exploration se records accidentally change na hon.
connection = engine.connect()
connection.execute(text("SET SESSION CHARACTERISTICS AS TRANSACTION READ ONLY"))
connection.execute(text("SET TIME ZONE 'Asia/Kolkata'"))
connection.commit()

print("Database connected.")

Database connected.


In [3]:
def query(sql, params=None):
    return pd.read_sql_query(
        text(sql),
        connection,
        params=params or {},
    )

In [4]:
display(query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
"""))

,table_name
0,appointments
1,conversations
2,doctors
3,messages


In [5]:
query("""
    SELECT 'doctors' AS table_name, COUNT(*) AS total FROM doctors
    UNION ALL
    SELECT 'conversations', COUNT(*) FROM conversations
    UNION ALL
    SELECT 'messages', COUNT(*) FROM messages
    UNION ALL
    SELECT 'appointments', COUNT(*) FROM appointments
""")

,table_name,total
0,doctors,100
1,conversations,20
2,messages,190
3,appointments,2


In [6]:
query("""
    SELECT id, name, specialty, location, rating, availability
    FROM doctors
    ORDER BY id
    LIMIT 10
""")

,id,name,specialty,location,rating,availability
0,1,Dr. Sarah Johnson,General Practice,Downtown Medical Center,4.8,Mon-Fri 9AM-5PM
1,2,Dr. Michael Chen,Cardiology,Heart Care Clinic,4.9,Mon-Fri 8AM-4PM
2,3,Dr. Emily Rodriguez,Internal Medicine,City Health Center,4.7,Mon-Fri 10AM-6PM
3,4,Dr. James Williams,Neurology,Brain & Spine Institute,4.8,Tue-Sat 9AM-5PM
4,5,Dr. Lisa Anderson,Dermatology,Skin Care Specialists,4.6,Mon-Fri 9AM-5PM
5,6,Dr. Robert Taylor,Orthopedics,Sports Medicine Center,4.9,Mon-Fri 8AM-6PM
6,7,Dr. Jennifer Lee,Gastroenterology,Digestive Health Clinic,4.7,Mon-Fri 9AM-4PM
7,8,Dr. David Martinez,ENT,Ear Nose Throat Center,4.8,Mon-Fri 9AM-5PM
8,9,Dr. Amanda White,Psychiatry,Mental Health Associates,4.7,Mon-Sat 10AM-7PM
9,10,Dr. Christopher Brown,Emergency Medicine,Emergency Care Center,4.8,24/7


In [7]:
data_conversation = query("""
    SELECT id, conversation_id, role, content, status, created_at
    FROM messages
    ORDER BY created_at DESC, id DESC
    LIMIT 20
""")

In [8]:
data_conversation[data_conversation['conversation_id'] == '42b2e58a-1fad-4d7c-91cc-a9835d2882e3']

,id,conversation_id,role,content,status,created_at
0,190,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,assistant,Please click Confirm booking on the appointmen...,completed,2026-10-18 05:50:10.076160+00:00
1,189,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,user,3:00 PM to 3:30 PM,completed,2026-10-18 05:50:08.359354+00:00
2,188,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,assistant,Dr. Lisa Anderson is available on 20th October...,completed,2026-10-18 05:49:56.774643+00:00
3,187,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,user,yes I am ok for 20th oct between 12-5pm,completed,2026-10-18 05:49:51.959584+00:00
4,186,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,assistant,"You have chosen Dr. Lisa Anderson, a dermatolo...",completed,2026-10-18 05:49:32.745496+00:00
5,185,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,user,I will prefer Dr Lisa,completed,2026-10-18 05:49:30.450471+00:00
6,184,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,assistant,I found two dermatologists for you:\n\n1. Dr. ...,completed,2026-10-18 05:49:18.300753+00:00
7,183,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,user,Find a dermatologist,completed,2026-10-18 05:49:12.989626+00:00


In [9]:
query("""
    SELECT
        a.id AS appointment_id,
        d.name AS doctor_name,
        a.conversation_id,
        a.starts_at,
        a.ends_at,
        a.created_at AS booked_at
    FROM appointments AS a
    JOIN doctors AS d ON d.id = a.doctor_id
    ORDER BY a.created_at DESC
    LIMIT 20
""")

,appointment_id,doctor_name,conversation_id,starts_at,ends_at,booked_at
0,b65a4b74-5e17-4c39-a803-15700d38b191,Dr. Sarah Johnson,42b2e58a-1fad-4d7c-91cc-a9835d2882e3,2026-10-20 09:30:00+00:00,2026-10-20 10:00:00+00:00,2026-10-18 05:50:50.771490+00:00
1,740b1ffe-43f1-4782-8847-6cd41ff2651e,Dr. Lisa Anderson,4c74982b-4368-49f3-a975-d8f08b50c216,2026-10-19 10:00:00+00:00,2026-10-19 10:30:00+00:00,2026-10-18 04:52:05.011249+00:00
